# Portfolioprüfung: Spam-Mail-Klassifikation

**Namen:** Noah, Marius, Leon  
**Modul:** Data Science / Künstliche Intelligenz  
**Semester:** 2. Semester  
**Abgabe:** 29.04.2026  

---

## Einleitung

Im Rahmen dieser Portfolioprüfung wird ein Datensatz mit E-Mails analysiert, die in die Kategorien **Spam** und **Nicht-Spam** eingeteilt sind. Ziel ist es, typische Merkmale von Spam-Nachrichten mithilfe von Natural Language Processing (NLP) zu identifizieren und darauf aufbauend Klassifikationsmodelle zu entwickeln und zu vergleichen.

Die Analyse orientiert sich am **CRISP-DM**-Prozess und umfasst folgende Schritte:

1. **Business Understanding** – Formulierung der Forschungsfrage und SMART-Ziele  
2. **Data Understanding** – Explorative Datenanalyse, Verteilungen, Wortfrequenzen  
3. **Data Preparation** – Textbereinigung mit Regex, Stopword-Entfernung, Tokenisierung  
4. **Modeling** – Vektorisierung (BoW, TF-IDF, N-Gramme) und KNN-Klassifikation  
5. **Evaluation** – Vergleich der Modelle anhand des F1-Scores, Zero-Shot-LLM  
6. **Deployment / Reflexion** – Kritische Reflexion der Ergebnisse  

---

## Forschungsfrage

> *Welche sprachlichen Merkmale in E-Mails können mithilfe grundlegender NLP-Methoden genutzt werden, um Spam- von Nicht-Spam-Nachrichten zu unterscheiden?*

---

## SMART-Zielsetzung

| Kriterium | Beschreibung |
|-----------|--------------|
| **S – Spezifisch** | Entwicklung einer Pipeline zur binären Klassifikation (Spam / Not Spam) auf einem E-Mail-CSV-Datensatz |
| **M – Messbar** | Bewertung über F1-Score (gewichtet) auf ungesehenen Validierungsdaten |
| **A – Erreichbar** | Umsetzbar mit Python, scikit-learn, NLTK und der Hugging Face-Bibliothek im Rahmen des Moduls |
| **R – Realistisch** | Der Datensatz liegt strukturiert vor und ist für eine binäre Klassifikation geeignet |
| **T – Terminiert** | Fertigstellung bis 29.04.2026 |

---

## Einordnung in CRISP-DM

```
Business Understanding → Data Understanding → Data Preparation
         ↓
      Modeling → Evaluation → (Deployment)
```

Das Notebook durchläuft alle Phasen iterativ: Erkenntnisse aus der Datenanalyse fließen in die Vorverarbeitung ein, und der Modellvergleich führt zur finalen Modellauswahl.

## 1  Vorbereitung der Arbeitsumgebung

Zu Beginn werden alle benötigten Python-Bibliotheken importiert. Diese decken das Einlesen der Daten, Textverarbeitung, Visualisierung und maschinelles Lernen ab.

In [ ]:
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (classification_report, accuracy_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)

## 2  Data Understanding

### 2.1  Dateneinlesung

Der E-Mail-Datensatz wird aus einer CSV-Datei eingelesen. Die Spalten `title` (Betreff), `text` (E-Mail-Inhalt) und `type` (Zielvariable: *spam* / *not spam*) werden erwartet.

In [ ]:
df = pd.read_csv("email_spam.csv")
df.head()

**Beobachtung:** Die ersten Zeilen zeigen sehr unterschiedliche Mailarten – von Anmeldecodes über Kundenservice-Antworten bis zu werblichen Inhalten. Die Struktur Feature / Label ist klar erkennbar, sodass eine überwachte Klassifikation grundsätzlich möglich ist.

### 2.2  Deskriptive Übersicht

In [ ]:
df.describe()

In [ ]:
df.info()

**Beobachtung:**
- **84 Zeilen, 3 Spalten** – kein einziger fehlender Wert.  
- **Zielvariable** `type` hat genau 2 Ausprägungen: *spam* und *not spam*.  
- **Klassenungleichgewicht:** `not spam` (58 Mails, ~69 %) ist die Mehrheitsklasse; `spam` hat nur 26 Einträge (~31 %).  
- Titel und Texte sind größtenteils individuell (78 bzw. 82 unterschiedliche Werte).  

Ein Modell könnte scheinbar gute Ergebnisse erzielen, wenn es stets die Mehrheitsklasse vorhersagt – daher wird der **F1-Score** als zentrale Metrik verwendet.

### 2.3  Verteilung der Zielvariable

Die Klassen *spam* und *not spam* werden gezählt und visualisiert.

In [ ]:
counts = df["type"].value_counts()

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(counts.index, counts.values, color=["#1D9E75", "#E24B4A"])
ax.bar_label(bars, fmt='%d', padding=3)
ax.set_title("Verteilung der Zielvariable")
ax.set_xlabel("Klasse")
ax.set_ylabel("Anzahl Mails")
ax.set_ylim(0, counts.max() + 8)
plt.tight_layout()
plt.show()

print(counts)
print(f"\nAnteil Spam:     {counts['spam'] / len(df):.1%}")
print(f"Anteil Not Spam: {counts['not spam'] / len(df):.1%}")

**Beobachtung:** Das Verhältnis beträgt ca. 69 % Not Spam zu 31 % Spam. Dieses Klassenungleichgewicht ist bei der Modellbewertung zu berücksichtigen.

### 2.4  Input- und Output-Variablen

| Rolle | Spalte | Beschreibung |
|-------|--------|--------------|
| **Input (X)** | `text` | Rohtext der E-Mail |
| **Input (X)** | `title` | Betreff der E-Mail (optional) |
| **Output (y)** | `type` | Klasse: *spam* oder *not spam* |

**Klasslabels:** $c_1$ = `not spam`, $c_2$ = `spam` → $k = 2$

**Machine-Learning-Problem:** Da die Zielvariable kategorisch und binär ist, handelt es sich um eine **Klassifikation**.

**Prediction vs. Inference:**
- *Prediction:* Für eine neue, unbekannte E-Mail soll vorhergesagt werden, ob sie Spam ist.  
- *Inference:* Welche Wörter (Features) haben den stärksten Einfluss auf die Spam-Klassifikation?

**Trainingsdaten-Beispiele:** $(x^{(1)}, y^{(1)}), \ldots, (x^{(n)}, y^{(n)})$ mit $n=84$, wobei $x^{(i)}$ der E-Mail-Text und $y^{(i)} \in \{\text{spam}, \text{not spam}\}$.

### 2.5  Beschreibung des Korpus

In [ ]:
df["tokens"] = df["text"].str.split()
all_words = np.concatenate(df["tokens"].values)

word_types     = pd.Series(all_words).nunique()
word_instances = len(all_words)

print(f"Word Types     (einzigartige Wörter): {word_types}")
print(f"Word Instances (Gesamtvorkommen):      {word_instances}")
print(f"Type-Token-Ratio (TTR):                {word_types / word_instances:.3f}")

**Beobachtung:** Die Type-Token-Ratio (TTR) gibt Auskunft über die lexikalische Vielfalt des Korpus. Ein höherer Wert bedeutet mehr einzigartige Wörter im Verhältnis zur Gesamtwortzahl.

## 3  Data Preparation

### 3.1  Wortfrequenzanalyse (ohne Stopword-Entfernung)

Die häufigsten Wörter im gesamten Korpus werden visualisiert.

In [ ]:
word_counts = pd.Series(all_words).value_counts().reset_index()
word_counts.columns = ["word", "count"]

top_n = 20
top_words = word_counts.head(top_n)

plt.figure(figsize=(10, 5))
plt.bar(top_words["word"], top_words["count"])
plt.title(f"Top {top_n} häufigste Wörter (alle Klassen)")
plt.xlabel("Wort")
plt.ylabel("Anzahl")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

**Beobachtung:** Die häufigsten Wörter sind englische Funktionswörter (*to*, *the*, *and*, *your*, *you*). Sie sind für die Klassifikation wenig aussagekräftig – eine Stopword-Entfernung ist notwendig.

### 3.2  Textbereinigung mit Regex und Stopword-Entfernung

Folgende Vorverarbeitungsschritte werden durchgeführt:

1. **Kleinschreibung** – Einheitliche Normalisierung (`lower()`)  
2. **Regex-Bereinigung** – Entfernung von URLs, E-Mail-Adressen, Sonderzeichen und Zahlen  
3. **Stopword-Entfernung** – Filterung englischer Funktionswörter via NLTK  
4. **Längenfilter** – Nur Wörter mit mehr als 2 Zeichen bleiben erhalten

In [ ]:
stop_words = set(stopwords.words("english"))

def clean_text(text):
    """Bereinigt einen Rohtext: Kleinschreibung, Regex, Stopwords."""
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)         # URLs entfernen
    text = re.sub(r"\S+@\S+", " ", text)                   # E-Mail-Adressen
    text = re.sub(r"[^a-z\s]", " ", text)                   # Sonderzeichen & Zahlen
    text = re.sub(r"\s+", " ", text).strip()                # Mehrfach-Leerzeichen
    # Stopwords + Längenfilter
    tokens = [w for w in text.split() if w not in stop_words and len(w) > 2]
    return " ".join(tokens)

df["clean"] = df["text"].apply(clean_text)
df[["text", "clean"]].head(3)

### 3.3  Wortfrequenzanalyse nach Klassen (nach Stopword-Entfernung)

Vergleich der häufigsten Wörter in Spam- und Not-Spam-Mails.

In [ ]:
df["clean_tokens"] = df["clean"].str.split()

n_spam = (df["type"] == "spam").sum()
n_ham  = (df["type"] == "not spam").sum()

spam_words_clean = np.concatenate(df[df["type"] == "spam"]["clean_tokens"].values)
ham_words_clean  = np.concatenate(df[df["type"] == "not spam"]["clean_tokens"].values)

spam_rel = (pd.Series(spam_words_clean).value_counts() / n_spam).head(15)
ham_rel  = (pd.Series(ham_words_clean).value_counts()  / n_ham).head(15)

x_max = max(spam_rel.values[0], ham_rel.values[0])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

ax1.barh(spam_rel.index[::-1], spam_rel.values[::-1], color="#E24B4A")
ax1.set_title("Top 15 Wörter – Spam", fontsize=13)
ax1.set_xlabel("Ø Vorkommen pro Mail")
ax1.set_xlim(0, x_max)

ax2.barh(ham_rel.index[::-1], ham_rel.values[::-1], color="#1D9E75")
ax2.set_title("Top 15 Wörter – Not Spam", fontsize=13)
ax2.set_xlabel("Ø Vorkommen pro Mail")
ax2.set_xlim(0, x_max)

plt.suptitle("Relative Wortfrequenzen nach Bereinigung", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Beobachtung:** Nach der Bereinigung zeigt sich deutlich mehr Trennschärfe:

- **Spam:** Aktionswörter wie *click*, *get*, *free*, *win* und Ansprache-Pronomen *your*, *our*  
- **Not Spam:** Sachliche Begriffe wie *company*, *customer*, *service*, *account*, *please*, *information*  

Diese Wortgruppen sind starke Indikatoren für die Klassifikation.

## 4  Modellierung

### 4.1  Datensplit: Training / Validierung / Test

Der Datensatz wird in drei Teile aufgeteilt:

- **Validierungsdaten (20 %):** Werden erst am Ende für den finalen, unvoreingenommenen Test verwendet. Keine Modellentscheidungen!  
- **Modelldaten (80 %):** Werden nochmals in **Trainingsdaten (80 %)** und **Testdaten (20 %)** aufgeteilt, um Hyperparameter (z. B. *k* bei KNN) zu optimieren.

```
Gesamt (84)
  ├── Validierung  (20 %, ~17 Mails)  → nur finaler Test
  └── Modell       (80 %, ~67 Mails)
        ├── Training  (80 %, ~54 Mails)  → Modell lernt
        └── Test      (20 %, ~13 Mails)  → Hyperparameter wählen
```

In [ ]:
X = df["clean"]
y = df["type"]

# Hold-out Validierungsset – wird erst ganz am Ende verwendet
X_model, X_val, y_model, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Innerer Train/Test-Split für Modellentscheidungen
X_train, X_test, y_train, y_test = train_test_split(
    X_model, y_model, test_size=0.2, random_state=42, stratify=y_model
)

print(f"Gesamt:       {len(X)} Mails")
print(f"Training:     {len(X_train)} Mails")
print(f"Test:         {len(X_test)} Mails")
print(f"Validierung:  {len(X_val)} Mails")

### 4.2  Hilfsfunktion für Modelltraining und -bewertung

Um verschiedene Vektorisierungs- und KNN-Varianten effizient zu vergleichen, wird eine gemeinsame Hilfsfunktion definiert.

In [ ]:
results = []  # Sammelt alle Ergebnisse für den späteren Vergleich

def train_and_evaluate(name, vectorizer, k, X_tr, y_tr, X_te, y_te, show_cm=False):
    """Trainiert einen KNN-Klassifikator und gibt den F1-Score zurück."""
    vec = vectorizer
    X_tr_vec = vec.fit_transform(X_tr)
    X_te_vec = vec.transform(X_te)

    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(X_tr_vec, y_tr)
    y_pred = clf.predict(X_te_vec)

    f1 = f1_score(y_te, y_pred, average='weighted')
    results.append({"Modell": name, "k": k, "F1 (Test)": round(f1, 4)})

    if show_cm:
        cm = confusion_matrix(y_te, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_)
        disp.plot(cmap="Blues")
        plt.title(f"Confusion Matrix – {name} (k={k})")
        plt.tight_layout()
        plt.show()
        print(classification_report(y_te, y_pred))

    return clf, vec, f1

### 4.3  KNN mit Bag-of-Words (Unigrams)

Der einfachste Vektorisierungsansatz: **Bag-of-Words** zählt, wie oft jedes Wort in einem Dokument vorkommt. Die Reihenfolge der Wörter wird ignoriert.

Verschiedene Werte für *k* werden getestet.

In [ ]:
for k in [1, 3, 5, 7]:
    train_and_evaluate(
        name=f"BoW",
        vectorizer=CountVectorizer(),
        k=k,
        X_tr=X_train, y_tr=y_train,
        X_te=X_test,  y_te=y_test
    )

# Bestes k für BoW anzeigen
bow_results = [r for r in results if r["Modell"] == "BoW"]
best_bow = max(bow_results, key=lambda r: r["F1 (Test)"])
print("Bestes BoW-Ergebnis:", best_bow)

### 4.4  KNN mit TF-IDF

**TF-IDF** (Term Frequency – Inverse Document Frequency) gewichtet Wörter, die in einem Dokument häufig, aber über den Korpus selten sind, höher. Das gibt seltenen, aber aussagekräftigen Wörtern mehr Gewicht.

In [ ]:
for k in [1, 3, 5, 7]:
    train_and_evaluate(
        name="TF-IDF",
        vectorizer=TfidfVectorizer(),
        k=k,
        X_tr=X_train, y_tr=y_train,
        X_te=X_test,  y_te=y_test
    )

tfidf_results = [r for r in results if r["Modell"] == "TF-IDF"]
best_tfidf = max(tfidf_results, key=lambda r: r["F1 (Test)"])
print("Bestes TF-IDF-Ergebnis:", best_tfidf)

### 4.5  KNN mit N-Gramm-Vektorisierung

**N-Gramme** erfassen Wortsequenzen und damit auch lokalen Kontext. Beispiele:
- Unigram (1,1): *free*
- Bigram (2,2): *free money*
- Trigram (3,3): *click here now*

Verschiedene N-Gramm-Bereiche werden getestet.

In [ ]:
for ngram in [(1,1), (1,2), (2,2), (1,3)]:
    for k in [3, 5]:
        label = f"N-gram {ngram}"
        train_and_evaluate(
            name=label,
            vectorizer=CountVectorizer(ngram_range=ngram),
            k=k,
            X_tr=X_train, y_tr=y_train,
            X_te=X_test,  y_te=y_test
        )

ngram_results = [r for r in results if r["Modell"].startswith("N-gram")]
best_ngram = max(ngram_results, key=lambda r: r["F1 (Test)"])
print("Bestes N-Gramm-Ergebnis:", best_ngram)

### 4.6  Modellvergleich

Alle trainierten Modelle werden anhand des F1-Scores auf den **Testdaten** visualisiert verglichen.

In [ ]:
df_results = pd.DataFrame(results)
df_results["Label"] = df_results["Modell"] + " k=" + df_results["k"].astype(str)

# Sortiert nach F1
df_results_sorted = df_results.sort_values("F1 (Test)", ascending=True)

plt.figure(figsize=(10, 7))
colors = ["#E24B4A" if f < 0.8 else "#1D9E75" for f in df_results_sorted["F1 (Test)"]]
plt.barh(df_results_sorted["Label"], df_results_sorted["F1 (Test)"], color=colors)
plt.axvline(0.8, color="gray", linestyle="--", label="F1 = 0.80")
plt.xlabel("F1-Score (gewichtet, Testdaten)")
plt.title("Modellvergleich: KNN mit verschiedenen Vektorisierungen")
plt.xlim(0, 1.05)
plt.legend()
plt.tight_layout()
plt.show()

print("\nAlle Ergebnisse:")
print(df_results.sort_values("F1 (Test)", ascending=False).to_string(index=False))

**Beobachtung:** Der Vergleich zeigt, welche Kombination aus Vektorisierung und *k* den höchsten F1-Score auf den Testdaten erzielt. Dieses Modell wird im nächsten Schritt auf den bisher ungesehenen Validierungsdaten geprüft.

### 4.7  Finaler Test auf Validierungsdaten

Das beste Modell (höchster F1-Score auf Testdaten) wird nun auf die **Validierungsdaten** angewendet – Daten, die das Modell noch nie gesehen hat und die bei keiner Modellentscheidung verwendet wurden.

In [ ]:
# Bestes Modell aus dem Vergleich ermitteln
best_row = df_results.loc[df_results["F1 (Test)"].idxmax()]
print(f"Bestes Modell: {best_row['Modell']} mit k={best_row['k']} → F1(Test) = {best_row['F1 (Test)']:.4f}")

# Bestes Modell neu trainieren (auf gesamten Modelldaten = Train + Test)
best_k = int(best_row["k"])

# Vektorisierer je nach Modellname wählen
if "TF-IDF" in best_row["Modell"]:
    best_vec = TfidfVectorizer()
elif "N-gram (1, 2)" in best_row["Modell"]:
    best_vec = CountVectorizer(ngram_range=(1,2))
elif "N-gram (2, 2)" in best_row["Modell"]:
    best_vec = CountVectorizer(ngram_range=(2,2))
elif "N-gram (1, 3)" in best_row["Modell"]:
    best_vec = CountVectorizer(ngram_range=(1,3))
else:
    best_vec = CountVectorizer()

# Training auf X_model (= X_train + X_test zusammen)
X_model_vec = best_vec.fit_transform(X_model)
best_clf = KNeighborsClassifier(n_neighbors=best_k)
best_clf.fit(X_model_vec, y_model)

# Test auf Validierungsdaten
X_val_vec = best_vec.transform(X_val)
y_val_pred = best_clf.predict(X_val_vec)

f1_val = f1_score(y_val, y_val_pred, average='weighted')
print(f"\nF1-Score auf Validierungsdaten: {f1_val:.4f}")
print("\nKlassifikationsbericht:")
print(classification_report(y_val, y_val_pred))

cm_val = confusion_matrix(y_val, y_val_pred)
disp_val = ConfusionMatrixDisplay(confusion_matrix=cm_val, display_labels=best_clf.classes_)
disp_val.plot(cmap="Blues")
plt.title(f"Confusion Matrix – Bestes Modell auf Validierungsdaten")
plt.tight_layout()
plt.show()

**Beobachtung:** Der F1-Score auf den Validierungsdaten zeigt, wie gut das Modell auf wirklich ungesehenen Daten generalisiert. Abweichungen vom Testdaten-F1 können auf Überanpassung (Overfitting) oder die geringe Datensatzgröße hinweisen.

### 4.8  Stichprobenprüfung: Vorhersage vs. echter Wert

Eine zufällige Stichprobe der Validierungsdaten wird manuell geprüft, um das Modellverhalten qualitativ nachzuvollziehen.

In [ ]:
sample_idx = X_val.index[:8]
sample_df = df.loc[sample_idx, ["title", "text", "type"]].copy()
sample_df["Vorhersage"] = y_val_pred[:8]
sample_df["Korrekt"] = sample_df["type"] == sample_df["Vorhersage"]

# Nur relevante Spalten und gekürzter Text
sample_df["text_kurz"] = sample_df["text"].str[:80] + "..."
print(sample_df[["title", "text_kurz", "type", "Vorhersage", "Korrekt"]].to_string(index=False))

**Beobachtung:** Die Stichprobe erlaubt eine qualitative Einschätzung: Falsch klassifizierte Mails geben Hinweise darauf, wo das Modell an seine Grenzen stößt – z. B. bei kurzen Texten oder Mails, deren Sprache nicht dem typischen Spam-Muster entspricht.

## 5  LLM: Zero-Shot-Classification

### 5.1  Ansatz

Als weiterer Vergleichsansatz wird **Zero-Shot-Classification** mit einem vortrainierten Large Language Model (LLM) eingesetzt. Dabei wird kein domänenspezifisches Training durchgeführt – das Modell nutzt nur sein allgemeines Sprachverständnis, um Texte den Labels zuzuordnen.

**Modell:** `facebook/bart-large-mnli` (Hugging Face)  
**Labels:** `["spam", "not spam"]`

---

### Warum könnte das LLM schlechter abschneiden als KNN?

Das Modell `bart-large-mnli` wurde auf dem **MultiNLI-Datensatz** (Williams et al., 2018) fine-getuned. MultiNLI ist eine Sammlung von 433.000 Satzpaaren aus **zehn Textgenres**, die alle aus dem Open American National Corpus (OANC) stammen:

| Genre | Beschreibung |
|-------|--------------|
| face-to-face | Transkripte von Zweipersonengesprächen |
| government | Regierungsberichte, Reden, Pressemitteilungen |
| letters | Philanthropische Spendenanfragen |
| 9/11 | Bericht der nationalen Kommission zu den Anschlägen |
| OUP | Sachbücher zu Textilindustrie und Kinderentwicklung |
| slate | Populärkultur-Artikel des Magazins *Slate* |
| telephone | Telefon-Transkripte |
| travel | Reiseführer |
| fiction | Belletristik |
| verbatim | Wörtlich transkribierte Gespräche |

**E-Mail-Spam kommt in keinem dieser Genres vor.** Das Modell hat während des Trainings nie gelernt, zwischen werblichen Spam-Phrasen (*"click here", "free prize", "win now"*) und seriösen geschäftlichen E-Mails zu unterscheiden – diese Domäne ist für das Modell völlig unbekannt.

Der Zero-Shot-Ansatz funktioniert, indem das Modell prüft, ob ein Text mit einem Label wie *"spam"* logisch übereinstimmt (Entailment). Da es jedoch nie auf E-Mail-Texte trainiert wurde, fehlt ihm das domänenspezifische Sprachverständnis für diesen Anwendungsfall. Ein auf E-Mails **fine-getuntes** Modell würde hier deutlich bessere Ergebnisse liefern.

**Vorteil Zero-Shot:** Kein Training nötig, sofort einsetzbar, kein annotierter Datensatz erforderlich.  
**Nachteil Zero-Shot:** Kein Domänenwissen, langsamere Inferenz, Ergebnisse hängen stark von der Label-Formulierung ab.

In [ ]:
from transformers import pipeline

# Zero-Shot-Classifier laden (lädt das Modell beim ersten Aufruf herunter)
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

candidate_labels = ["spam", "not spam"]

def predict_zero_shot(text):
    """Gibt das wahrscheinlichste Label zurück."""
    result = classifier(text[:512], candidate_labels=candidate_labels)
    return result["labels"][0]  # höchste Wahrscheinlichkeit

print("Zero-Shot-Classifier geladen. Starte Klassifikation...")

In [ ]:
# Klassifikation auf Validierungsdaten (kann einige Minuten dauern)
y_llm_pred = X_val.apply(predict_zero_shot).values

f1_llm = f1_score(y_val, y_llm_pred, average='weighted')
print(f"F1-Score LLM (Zero-Shot): {f1_llm:.4f}")
print("\nKlassifikationsbericht:")
print(classification_report(y_val, y_llm_pred))

cm_llm = confusion_matrix(y_val, y_llm_pred)
disp_llm = ConfusionMatrixDisplay(confusion_matrix=cm_llm, display_labels=["not spam", "spam"])
disp_llm.plot(cmap="Purples")
plt.title("Confusion Matrix – LLM Zero-Shot")
plt.tight_layout()
plt.show()

### 5.2  Gesamtvergleich: KNN-Modelle vs. LLM

Alle KNN-Modelle werden anhand ihres F1-Scores auf den **Testdaten** verglichen – ergänzt um den LLM-F1-Score, der ebenfalls auf den Testdaten berechnet wird.

In [ ]:
# LLM auch auf Testdaten auswerten
y_llm_test_pred = X_test.apply(predict_zero_shot).values
f1_llm_test = f1_score(y_test, y_llm_test_pred, average='weighted')

# Alle KNN-Ergebnisse (bereits auf Testdaten berechnet) + LLM zusammenführen
comp_df = pd.DataFrame(results)  # enthält Modell, k, F1 (Test)
comp_df["Label"] = comp_df["Modell"] + " k=" + comp_df["k"].astype(str)

# LLM-Zeile ergänzen
llm_row = pd.DataFrame([{"Modell": "LLM Zero-Shot", "k": "-", "F1 (Test)": round(f1_llm_test, 4), "Label": "LLM Zero-Shot (bart-large-mnli)"}])
comp_df = pd.concat([comp_df, llm_row], ignore_index=True)

comp_df_sorted = comp_df.sort_values("F1 (Test)", ascending=True)

plt.figure(figsize=(10, 7))
colors = ["#7B5EA7" if "LLM" in str(lbl) else "#1D9E75" for lbl in comp_df_sorted["Label"]]
plt.barh(comp_df_sorted["Label"], comp_df_sorted["F1 (Test)"], color=colors)
plt.xlabel("F1-Score (gewichtet, Testdaten)")
plt.title("Gesamtvergleich aller Modelle auf Testdaten")
plt.xlim(0, 1.05)
plt.tight_layout()
plt.show()

print(comp_df.sort_values("F1 (Test)", ascending=False)[["Label", "F1 (Test)"]].to_string(index=False))

**Beobachtung:** Der Vergleich auf den Testdaten zeigt die Stärken und Schwächen beider Ansätze:

- **KNN-Modelle** sind schnell, interpretierbar und für kleine Datensätze gut geeignet. Ihre Leistung hängt stark von der Wahl der Vektorisierung und von *k* ab.  
- **LLM Zero-Shot** benötigt kein Training und profitiert von umfangreichem Vorwissen. Bei kurzen, spezifischen Texten kann das Modell jedoch unsicher sein.  
- Beide Ansätze haben ihre Berechtigung – für Produktionssysteme wäre ein fine-getuntes LLM die stärkste Option, aber auch ressourcenintensiver.

## 6  Kritische Reflexion

### Ergebnisse und Erwartungen

Die KNN-Modelle erreichen auf einem so kleinen Datensatz (84 Mails) bereits solide F1-Scores, was teils auf die recht klaren sprachlichen Muster in Spam-Mails zurückzuführen ist (*click*, *free*, *win* vs. *company*, *customer*, *service*). Die Ergebnisse lagen im erwarteten Bereich, sind aber mit Vorsicht zu interpretieren: Kleine Datensätze führen zu hoher Varianz in den Metriken – schon eine einzelne falsch klassifizierte Mail verschiebt den F1-Score merklich.

### Grenzen und Einschränkungen

| Einschränkung | Beschreibung |
|---------------|--------------|
| **Datensatzgröße** | 84 Mails sind für robuste ML-Aussagen sehr wenig. Ergebnisse können stark von der zufälligen Aufteilung abhängen. |
| **Klassenungleichgewicht** | 69 % Not Spam vs. 31 % Spam – Precision/Recall für Spam ist tendenziell schlechter. Maßnahmen (SMOTE, class_weight) wurden aus Zeitgründen nicht umgesetzt. |
| **LLM-Inferenzzeit** | Zero-Shot-Classification mit `bart-large-mnli` ist signifikant langsamer als KNN. Auf beschränkter Hardware ist das ein praktischer Nachteil. |
| **Keine Lemmatisierung** | Wörter wie *clicking*, *clicked*, *clicks* werden als verschiedene Features behandelt. Lemmatisierung (spaCy) hätte die Featuregröße reduziert. |
| **BPE-Tokenisierung** | Wurde exploriert, aber für das finale Modell nicht verwendet, da die einfacheren BoW/TF-IDF-Ansätze vergleichbare Ergebnisse lieferten. |
| **Überanpassung** | KNN mit k=1 neigt zu Overfitting, was sich in großen Unterschieden zwischen Test- und Validierungs-F1 zeigen kann. |

### Annahmen

- Die bereinigten Texte (`clean`) sind ausreichend für die Klassifikation; der Betreff (`title`) wurde vernachlässigt.  
- Englischsprachige Stopword-Listen sind für diesen Datensatz geeignet.  
- Die Random-Seed-Fixierung (`random_state=42`) sorgt für Reproduzierbarkeit, schützt aber nicht vor Overfitting auf die konkrete Aufteilung.

### Vergleich mit der Literatur

Klassische Spam-Klassifikationsansätze (z. B. *Naive Bayes* auf dem Enron-Datensatz) erzielen F1-Scores von 0.95+, profitieren aber von Tausenden von Trainingsbeispielen. Für Miniaturkorpora wie hier ist KNN mit TF-IDF ein etablierter Baseline-Ansatz (vgl. Metsis et al., 2006). LLMs zeigen in Zero-Shot-Settings vielversprechende Ergebnisse, benötigen aber auch keine Trainingsdaten – ihr Einsatz ist vor allem dann sinnvoll, wenn annotierte Daten fehlen.

### Fazit

Das Projektziel – einen E-Mail-Datensatz mit NLP-Methoden zu analysieren und Spam zu klassifizieren – wurde erreicht. Für eine produktionsreife Lösung wäre ein größerer, ausgewogener Datensatz und ein fine-getuntes Sprachmodell der nächste Schritt.